## Day 8: Advanced OOP 
Today we go deeper into Object Oriented Programming. We'll explore powerful concepts like composition, polymorphism with duck typing, different types of methods and variables, decorators, dataclasses, and best practices for designing clean classes.

These tools help write more flexible, maintainable, and Pythonic code for real-world applications like building a complete School Management System, an E-commerce platform, or a Game Engine.

### Encapsulation Deep Dive: Composition
Composition is a "has-a" relationship. Instead of inheriting everything (is-a), we build complex objects by containing simpler ones. This is often preferred because it's more flexible — we can swap parts easily without affecting the whole hierarchy.

In [1]:
class Engine:
    def __init__(self, horsepower, fuel_type="petrol"):
        self.horsepower = horsepower
        self.fuel_type = fuel_type
    
    def start(self):
        return f"{self.fuel_type.capitalize()} engine with {self.horsepower} hp roaring to life!"

class Car:
    def __init__(self, model, horsepower):
        self.model = model
        self.engine = Engine(horsepower)  # Composition: Car HAS an Engine
        self.speed = 0
    
    def start_car(self):
        return f"{self.model} - {self.engine.start()}"
    
    def accelerate(self, increase):
        self.speed += increase
        return f"Speed now: {self.speed} km/h"

# Usage
my_car = Car("Tesla Model S", 670)
print(my_car.start_car())
print(my_car.accelerate(30))

Tesla Model S - Petrol engine with 670 hp roaring to life!
Speed now: 30 km/h


### Dynamic Extension
Python's dynamic nature lets us add attributes or even methods to individual objects at runtime. This is powerful for one-off customizations but should be used sparingly to keep code readable.

In [2]:
class Student:
    def __init__(self, name, roll_no):
        self.name = name
        self.roll_no = roll_no

s1 = Student("Priya Sharma", 101)
s1.marks = 92          # Added dynamically
s1.subjects = ["Math", "Physics"]

s2 = Student("Rahul Verma", 102)

print(f"{s1.name} has marks: {s1.marks}")
print("Does s2 have marks attribute?", hasattr(s2, 'marks'))

Priya Sharma has marks: 92
Does s2 have marks attribute? False


In [3]:
import types

def celebrate_birthday(self):
    print(f"Happy Birthday {self.name}!")

# Bind the function as a method to only this instance
s1.celebrate = types.MethodType(celebrate_birthday, s1)
s1.celebrate()

Happy Birthday Priya Sharma!


### Polymorphism & Duck Typing
Polymorphism allows the same interface to work with different object types. In Python, we rely heavily on **Duck Typing**: "If it walks like a duck and quacks like a duck, then it is a duck." No need for inheritance — just having the required methods is enough.

In [4]:
class Dog:
    def speak(self):
        return "Woof!"

class Cat:
    def speak(self):
        return "Meow!"

class Robot:
    def speak(self):  # No inheritance needed
        return "Beep boop!"

def make_animal_talk(animal):
    print(animal.speak())

for creature in [Dog(), Cat(), Robot()]:
    make_animal_talk(creature)

Woof!
Meow!
Beep boop!


### Class Variables vs Instance Variables
Class variables are shared across all instances. Instance variables are unique to each object.

In [5]:
class Student:
    school_name = "Green Valley International School"  # Class variable
    total_students = 0
    
    def __init__(self, name, grade):
        self.name = name          # Instance variable
        self.grade = grade
        Student.total_students += 1

s1 = Student("Priya", "10th")
s2 = Student("Rahul", "11th")

print(s1.school_name)  # Shared
print(Student.total_students)

Green Valley International School
2


### Types of Methods
#### 1. Instance Methods (most common)

In [6]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance  # Private by convention
    
    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount
            return f"Deposited ₹{amount}. New balance: ₹{self.__balance}"
    
    def withdraw(self, amount):
        if amount > self.__balance:
            return "Insufficient funds!"
        self.__balance -= amount
        return f"Withdrew ₹{amount}. Remaining: ₹{self.__balance}"

acc = BankAccount("Priya Sharma", 5000)
print(acc.deposit(2000))
print(acc.withdraw(1500))

Deposited ₹2000. New balance: ₹7000
Withdrew ₹1500. Remaining: ₹5500


#### 2. Class Methods - Use `@classmethod`

In [7]:
class Student:
    total_students = 0
    
    def __init__(self, name):
        self.name = name
        Student.total_students += 1
    
    @classmethod
    def from_csv(cls, data_string):
        """Alternative constructor from string like 'name,grade'"""
        name, grade = data_string.split(",")
        return cls(name)
    
    @classmethod
    def get_total(cls):
        return cls.total_students

s = Student.from_csv("Aarav,10th")
print(s.name, "- Total students:", Student.get_total())

Aarav - Total students: 1


#### 3. Static Methods - Use `@staticmethod`

In [8]:
class MathUtils:
    @staticmethod
    def is_even(number):
        return number % 2 == 0
    
    @staticmethod
    def percentage(obtained, total):
        return (obtained / total) * 100

print(MathUtils.is_even(42))
print(f"Score: {MathUtils.percentage(87, 100):.1f}%")

True
Score: 87.0%


### Decorators
Decorators allow us to wrap functions or methods to add extra functionality (logging, timing, validation, etc.) without modifying the original code.

In [9]:
def timer(func):
    import time
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end-start:.4f} seconds")
        return result
    return wrapper

@timer
def slow_operation(n):
    total = 0
    for i in range(n):
        total += i
    return total

print("Result:", slow_operation(100000))

slow_operation took 0.0022 seconds
Result: 4999950000


### Dataclasses
`@dataclass` reduces boilerplate for data-holding classes by auto-generating `__init__`, `__repr__`, `__eq__`, etc.

In [10]:
from dataclasses import dataclass, field
from datetime import date

@dataclass
class Product:
    name: str
    price: float
    stock: int = 0
    added_date: date = field(default_factory=date.today)
    
    def __post_init__(self):
        if self.price < 0:
            raise ValueError("Price cannot be negative!")

p1 = Product("Wireless Headphones", 2999.99, 50)
p2 = Product("Smart Watch", 4499.50, 30)

print(p1)
print(p1 == Product("Wireless Headphones", 2999.99, 50))  # True thanks to auto __eq__

Product(name='Wireless Headphones', price=2999.99, stock=50, added_date=datetime.date(2026, 6, 16))
True


### Best Practices: How Long Should a Class Be?
- Follow **Single Responsibility Principle** (SRP): A class should have only one reason to change.
- Keep classes focused and small.
- Use composition to combine smaller classes.
- Methods should be short and do one thing well.
- Good names are crucial — avoid god classes like `Manager` that do everything.

### Recap of Advanced OOP
- **Composition** > Inheritance for flexibility
- **Duck Typing** makes Python very flexible
- Use **Class Methods** for factories, **Static Methods** for utilities
- **Decorators** add powerful cross-cutting concerns
- **Dataclasses** save time on simple data containers
- Design for clarity and single responsibility